In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("lab").getOrCreate()

tripData = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y H:m") \
    .csv("/home/jovyan/work/trips.csv")

stationData = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y") \
    .csv("/home/jovyan/work/stations.csv")

In [8]:
tripData.show(5)
stationData.show(5)
tripData.printSchema()
stationData.printSchema()

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|2013-08-29 14:13:00|South Van Ness at...|              66|2013-08-29 14:14:00|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|      70|2013-08-29 14:42:00|  San Jose City Hall|              10|2013-08-29 14:43:00|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|    

# Задание 1
Найти велосипед с максимальным временем пробега.

In [15]:
task1 = tripData.orderBy(col("duration").desc())

task1.select(
    "bike_id",
    "duration",
    "start_station_name",
    "end_station_name",
    "start_date",
    "end_date"
).show(1, truncate=False)

+-------+--------+------------------------+----------------+-------------------+-------------------+
|bike_id|duration|start_station_name      |end_station_name|start_date         |end_date           |
+-------+--------+------------------------+----------------+-------------------+-------------------+
|535    |17270400|South Van Ness at Market|2nd at Folsom   |2014-12-06 21:59:00|2015-06-24 20:18:00|
+-------+--------+------------------------+----------------+-------------------+-------------------+
only showing top 1 row



# Задание 2 
Найти наибольшее геодезическое расстояние между станциями.

In [10]:
s1 = stationData.select(
    col("id").alias("id1"),
    col("name").alias("station1"),
    col("lat").alias("lat1"),
    col("long").alias("lon1")
)

s2 = stationData.select(
    col("id").alias("id2"),
    col("name").alias("station2"),
    col("lat").alias("lat2"),
    col("long").alias("lon2")
)

pairs = s1.crossJoin(s2).filter(col("id1") < col("id2"))

task2 = pairs.withColumn(
    "distance_km",
    lit(6371.0) * 2 * asin(sqrt(
        pow(sin((radians(col("lat2")) - radians(col("lat1"))) / 2), 2) +
        cos(radians(col("lat1"))) * cos(radians(col("lat2"))) *
        pow(sin((radians(col("lon2")) - radians(col("lon1"))) / 2), 2)
    ))
).orderBy(col("distance_km").desc())

task2.select("station1", "station2", "distance_km").show(1, truncate=False)

+--------------------------+----------------------+-----------------+
|station1                  |station2              |distance_km      |
+--------------------------+----------------------+-----------------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|69.92087595428183|
+--------------------------+----------------------+-----------------+
only showing top 1 row



# Задание 3
Найти путь велосипеда с максимальным временем пробега через станции.

In [11]:
task3 = tripData.withColumn(
    "path",
    concat_ws(" -> ", col("start_station_name"), col("end_station_name"))
).orderBy(col("duration").desc())

task3.select(
    "bike_id",
    "duration",
    "path",
    "start_date",
    "end_date"
).show(1, truncate=False)

+-------+--------+-----------------------------------------+-------------------+-------------------+
|bike_id|duration|path                                     |start_date         |end_date           |
+-------+--------+-----------------------------------------+-------------------+-------------------+
|535    |17270400|South Van Ness at Market -> 2nd at Folsom|2014-12-06 21:59:00|2015-06-24 20:18:00|
+-------+--------+-----------------------------------------+-------------------+-------------------+
only showing top 1 row



# Задание 4
Найти количество велосипедов в системе.

In [12]:
task4 = tripData.select("bike_id").distinct().count()
print("Количество велосипедов в системе:", task4)

Количество велосипедов в системе: 700


# Задание 5
Найти пользователей потративших на поездки более 3 часов.

In [13]:
task5 = tripData.filter(col("zip_code").isNotNull()) \
    .groupBy("zip_code") \
    .agg(sum("duration").alias("total_duration_sec")) \
    .filter(col("total_duration_sec") > 3 * 60 * 60) \
    .orderBy(col("total_duration_sec").desc())

task5.show(truncate=False)

+--------+------------------+
|zip_code|total_duration_sec|
+--------+------------------+
|94107   |49757162          |
|nil     |45725550          |
|94105   |25596128          |
|94133   |21637675          |
|94102   |19128021          |
|94103   |19127388          |
|95531   |17270400          |
|94111   |14244997          |
|95112   |12742370          |
|94109   |12057128          |
|94040   |7807926           |
|94110   |7421936           |
|94117   |6901313           |
|94301   |6590378           |
|94041   |6276284           |
|94158   |6248167           |
|94306   |5550643           |
|94025   |5178237           |
|94108   |5127562           |
|94611   |5014906           |
+--------+------------------+
only showing top 20 rows

